# Turing's "Computing Machinery and Intelligence" (1950) — Implementation
# 튜링의 "컴퓨팅 기계와 지능" (1950) — 구현

This paper is philosophical, with no algorithms in the traditional sense.
However, Turing's key ideas — the Imitation Game, discrete-state machines,
universality, and learning machines — can all be demonstrated in code.

이 논문은 철학 논문이지만, Turing의 핵심 아이디어들 — Imitation Game,
discrete-state machine, universality, learning machine — 은 모두 코드로 시연할 수 있습니다.

**Contents / 목차:**
1. Discrete-State Machine Simulation — 이산 상태 기계 시뮬레이션
2. Turing Machine from Scratch — Turing Machine 직접 구현
3. Universality Demonstration — 범용성 시연
4. The Imitation Game: Statistical Framework — Imitation Game 통계적 프레임워크
5. Child Machine: Learning with Reward/Punishment — 어린이 기계: 보상/벌 학습
6. Connection to Modern AI — 현대 AI와의 연결

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

## Part 1: Discrete-State Machine / 이산 상태 기계

Turing defines digital computers as a special case of **discrete-state machines** —
systems with a finite number of distinguishable states that transition deterministically.

Turing은 디지털 컴퓨터를 **discrete-state machine**의 특수한 경우로 정의합니다 —
유한한 수의 구별 가능한 상태를 가지고 결정론적으로 전이하는 시스템입니다.

Here we implement a simple finite-state machine: a turnstile (회전문) and then
show how its behavior is fully determined by its state table.

> "Given the state at some moment... it is possible to predict all future states."
> — Turing, Section 4

In [ ]:
class DiscreteStateMachine:
    """A simple finite-state machine (Turing's 'discrete-state machine')."""

    def __init__(self, states, initial_state, transition_table):
        """Initialize a discrete-state machine.

        Args:
            states: Set of possible states.
            initial_state: Starting state.
            transition_table: Dict of {(state, input): next_state}.
        """
        self.states = states
        self.state = initial_state
        self.transitions = transition_table
        self.history = [initial_state]

    def step(self, input_symbol):
        """Process one input and transition to next state."""
        key = (self.state, input_symbol)
        if key not in self.transitions:
            raise ValueError(f"No transition for state={self.state}, input={input_symbol}")
        self.state = self.transitions[key]
        self.history.append(self.state)
        return self.state

    def run(self, input_sequence):
        """Process a sequence of inputs."""
        return [self.step(inp) for inp in input_sequence]


# Example: Turnstile (회전문)
# States: LOCKED, UNLOCKED
# Inputs: COIN (동전), PUSH (밀기)
turnstile = DiscreteStateMachine(
    states={"LOCKED", "UNLOCKED"},
    initial_state="LOCKED",
    transition_table={
        ("LOCKED", "COIN"): "UNLOCKED",
        ("LOCKED", "PUSH"): "LOCKED",      # push locked door = stays locked
        ("UNLOCKED", "COIN"): "UNLOCKED",   # extra coin = stays unlocked
        ("UNLOCKED", "PUSH"): "LOCKED",     # push through = locks again
    }
)

# Run a sequence
inputs = ["PUSH", "COIN", "PUSH", "PUSH", "COIN", "COIN", "PUSH"]
outputs = turnstile.run(inputs)

print("Turnstile Discrete-State Machine / 회전문 이산 상태 기계")
print("=" * 55)
print(f"{'Step':>4}  {'Input':>8}  →  {'State':<10}")
print("-" * 55)
print(f"{'0':>4}  {'(start)':>8}  →  {'LOCKED':<10}")
for i, (inp, out) in enumerate(zip(inputs, outputs)):
    print(f"{i+1:>4}  {inp:>8}  →  {out:<10}")

print("\nKey insight (핵심 통찰):")
print("Given the initial state and input sequence,")
print("ALL future states are perfectly determined.")
print("This is what Turing means by 'discrete-state machine'.")

## Part 2: Turing Machine from Scratch / Turing Machine 직접 구현

Turing's 1936 abstract machine: an infinite tape, a read/write head, and
a finite set of transition rules. Despite its simplicity, it can compute
**anything computable** (Church-Turing thesis).

Turing의 1936년 추상 기계: 무한한 테이프, 읽기/쓰기 헤드, 유한한 전이 규칙 집합.
단순하지만 **계산 가능한 모든 것**을 계산할 수 있습니다 (Church-Turing thesis).

We implement a Turing Machine and run two programs:
1. Binary increment (이진수 +1)
2. Simple pattern recognition (단순 패턴 인식)

In [ ]:
class TuringMachine:
    """A Turing Machine implementation from scratch.

    Components (Turing's 3 elements mapped):
        - tape (Store): infinite tape of symbols
        - head position + state (Control): determines next action
        - transition rules (Executive Unit): performs read/write/move
    """

    BLANK = "_"

    def __init__(self, transition_rules, initial_state, accept_states, tape=None):
        """Initialize a Turing Machine.

        Args:
            transition_rules: Dict of {(state, symbol): (new_symbol, direction, new_state)}.
                direction is 'L' (left) or 'R' (right).
            initial_state: Starting state name.
            accept_states: Set of accepting/halting states.
            tape: Initial tape contents as string (default: blank).
        """
        self.rules = transition_rules
        self.state = initial_state
        self.accept_states = accept_states
        self.tape = defaultdict(lambda: self.BLANK)
        self.head = 0
        self.steps = 0
        self.history = []

        if tape:
            for i, ch in enumerate(tape):
                self.tape[i] = ch
            self.head = len(tape) - 1  # start at rightmost symbol

    def step(self):
        """Execute one step of the Turing Machine."""
        symbol = self.tape[self.head]
        key = (self.state, symbol)

        if key not in self.rules:
            return False  # halt (no rule)

        new_symbol, direction, new_state = self.rules[key]

        # Record history
        self.history.append({
            "step": self.steps,
            "state": self.state,
            "head": self.head,
            "read": symbol,
            "write": new_symbol,
            "move": direction,
            "new_state": new_state,
        })

        # Execute
        self.tape[self.head] = new_symbol
        self.head += 1 if direction == "R" else -1
        self.state = new_state
        self.steps += 1

        return self.state not in self.accept_states

    def run(self, max_steps=1000):
        """Run until halting or max_steps."""
        while self.step() and self.steps < max_steps:
            pass
        return self.get_tape()

    def get_tape(self):
        """Return tape contents as string."""
        if not self.tape:
            return self.BLANK
        min_pos = min(self.tape.keys())
        max_pos = max(self.tape.keys())
        return "".join(self.tape[i] for i in range(min_pos, max_pos + 1)).strip("_")


# Program 1: Binary Increment (이진수 +1)
# Input: binary number, head starts at rightmost digit
# Rules: carry propagation from right to left
binary_increment = TuringMachine(
    transition_rules={
        # State "carry": propagate carry to the left
        ("carry", "1"): ("0", "L", "carry"),   # 1+1=0, carry continues
        ("carry", "0"): ("1", "L", "done"),     # 0+1=1, no more carry
        ("carry", "_"): ("1", "R", "done"),     # overflow: write new 1
    },
    initial_state="carry",
    accept_states={"done"},
    tape="1011"  # binary 11
)

result = binary_increment.run()
print("Program 1: Binary Increment / 이진수 +1")
print("=" * 50)
print(f"  Input:  1011  (decimal {int('1011', 2)})")
print(f"  Output: {result}  (decimal {int(result, 2)})")
print(f"  Steps:  {binary_increment.steps}")
print()

# Show step-by-step
print("Step-by-step trace / 단계별 추적:")
print(f"  {'Step':>4}  {'State':>6}  {'Read':>4}  →  {'Write':>5}  {'Move':>4}  {'Next':>6}")
print("  " + "-" * 46)
for h in binary_increment.history:
    print(f"  {h['step']:>4}  {h['state']:>6}  {h['read']:>4}  →  "
          f"{h['write']:>5}  {h['move']:>4}  {h['new_state']:>6}")

# Program 2: Test multiple inputs
print("\n\nBinary increment on various inputs / 다양한 입력에 대한 이진수 +1:")
print("-" * 40)
for binary_str in ["0", "1", "101", "111", "1111"]:
    tm = TuringMachine(
        transition_rules={
            ("carry", "1"): ("0", "L", "carry"),
            ("carry", "0"): ("1", "L", "done"),
            ("carry", "_"): ("1", "R", "done"),
        },
        initial_state="carry",
        accept_states={"done"},
        tape=binary_str
    )
    result = tm.run()
    dec_in = int(binary_str, 2)
    dec_out = int(result, 2)
    print(f"  {binary_str:>5} ({dec_in:>2})  →  {result:>5} ({dec_out:>2})  [{tm.steps} steps]")

## Part 3: Universality — One Machine Simulates Another / 범용성 시연

Turing's key argument in Section 5: a **universal** digital computer can simulate
any other discrete-state machine, given enough storage.

This means we don't need a special "thinking machine" — just the right program
on a general-purpose computer.

> "These special computers may then be considered to be approximations
> to a universal computer." — Turing, Section 5

We demonstrate this by building a **Universal Turing Machine** that reads
another machine's rules from its tape and executes them.

Section 5의 핵심 논증: **범용** 디지털 컴퓨터는 충분한 저장 공간만 있으면
어떤 discrete-state machine이든 시뮬레이션할 수 있다.

즉, 특별한 "생각하는 기계"가 필요 없다 — 범용 컴퓨터에 올바른 프로그램만 있으면 된다.

In [ ]:
class UniversalTuringMachine:
    """A Universal Turing Machine that simulates any other TM.

    This is Turing's universality argument in code: ONE machine can
    simulate ANY other machine by reading its description (rules) as data.
    """

    def __init__(self):
        """Initialize the Universal TM (no specific program yet)."""
        pass

    def simulate(self, rules, initial_state, accept_states, tape_str, max_steps=1000):
        """Simulate an arbitrary Turing Machine given its description.

        Args:
            rules: Transition rules as dict {(state, symbol): (write, dir, next_state)}.
            initial_state: The simulated machine's initial state.
            accept_states: The simulated machine's halting states.
            tape_str: Initial tape contents.
            max_steps: Safety limit.

        Returns:
            Final tape contents.
        """
        # The UTM treats the OTHER machine's rules as DATA
        tape = defaultdict(lambda: "_")
        for i, ch in enumerate(tape_str):
            tape[i] = ch

        state = initial_state
        head = len(tape_str) - 1
        steps = 0

        while steps < max_steps:
            symbol = tape[head]
            key = (state, symbol)

            if key not in rules or state in accept_states:
                break

            write, direction, next_state = rules[key]
            tape[head] = write
            head += 1 if direction == "R" else -1
            state = next_state
            steps += 1

        min_pos = min(tape.keys())
        max_pos = max(tape.keys())
        return "".join(tape[i] for i in range(min_pos, max_pos + 1)).strip("_")


# The Universal Machine
utm = UniversalTuringMachine()

# Machine A: Binary increment (same as before)
rules_A = {
    ("carry", "1"): ("0", "L", "carry"),
    ("carry", "0"): ("1", "L", "done"),
    ("carry", "_"): ("1", "R", "done"),
}

# Machine B: Invert bits (0→1, 1→0)
rules_B = {
    ("invert", "0"): ("1", "L", "invert"),
    ("invert", "1"): ("0", "L", "invert"),
    ("invert", "_"): ("_", "R", "done"),
}

# ONE universal machine simulates BOTH
print("Universal Turing Machine / 범용 Turing 기계")
print("=" * 55)
print("The SAME machine simulates two DIFFERENT programs:\n")

print("Machine A: Binary Increment / 이진수 +1")
for tape in ["101", "111", "1001"]:
    result = utm.simulate(rules_A, "carry", {"done"}, tape)
    print(f"  {tape} ({int(tape,2):>2}) → {result} ({int(result,2):>2})")

print("\nMachine B: Bit Inversion / 비트 반전")
for tape in ["101", "111", "1001"]:
    result = utm.simulate(rules_B, "invert", {"done"}, tape)
    print(f"  {tape} → {result}")

print("\n" + "=" * 55)
print("Key insight (핵심 통찰):")
print("ONE machine, DIFFERENT programs → DIFFERENT behaviors.")
print("This is universality: the machine is general-purpose.")
print("Intelligence is therefore a SOFTWARE problem, not hardware.")

## Part 4: The Imitation Game — Statistical Framework / 통계적 프레임워크

Turing predicted that by ~2000, a machine could play the Imitation Game well
enough that "an average interrogator will not have more than 70% chance of
making the right identification after five minutes of questioning."

We model this statistically: if the interrogator's accuracy drops to chance
level (50%), the machine is indistinguishable from a human.

Turing은 ~2000년까지 기계가 Imitation Game을 잘 수행하여 "평균적인 심문자가
5분 질문 후 올바른 식별을 할 확률이 70%를 넘지 못할 것"이라 예측했습니다.

In [ ]:
def simulate_imitation_game():
    """Simulate the Imitation Game and visualize Turing's prediction."""
    np.random.seed(42)

    # Model: interrogator has some base accuracy that depends on machine quality
    # As machines improve over decades, interrogator accuracy drops toward 50%

    # Historical timeline of "machine quality" (0=no AI, 1=human-level)
    years = np.arange(1950, 2030)

    # Sigmoid growth of machine capability
    machine_quality = 1 / (1 + np.exp(-0.08 * (years - 2005)))

    # Interrogator accuracy: starts at ~100% (easy to detect), drops toward 50%
    interrogator_accuracy = 0.50 + 0.50 * (1 - machine_quality)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: Interrogator accuracy over time
    ax1 = axes[0]
    ax1.plot(years, interrogator_accuracy * 100, "steelblue", lw=3)
    ax1.axhline(70, color="red", ls="--", lw=2, label="Turing's threshold (70%)")
    ax1.axhline(50, color="gray", ls=":", lw=1, label="Chance level (50%)")

    # Mark Turing's prediction
    ax1.axvline(2000, color="green", ls="--", alpha=0.5)
    ax1.annotate("Turing's prediction\n(~2000, ≤70%)",
                 xy=(2000, 70), xytext=(1970, 85),
                 arrowprops=dict(arrowstyle="->", color="green"),
                 fontsize=10, color="green")

    # Mark key milestones
    milestones = [
        (1966, 92, "ELIZA"),
        (1991, 88, "Loebner\nPrize"),
        (2014, 73, "Eugene\nGoostman"),
        (2022, 58, "ChatGPT"),
    ]
    for yr, acc, label in milestones:
        ax1.plot(yr, acc, "ko", ms=8)
        ax1.annotate(label, (yr, acc - 4), ha="center", fontsize=9)

    ax1.set_xlabel("Year / 연도")
    ax1.set_ylabel("Interrogator accuracy (%) / 심문자 정확도 (%)")
    ax1.set_title("Imitation Game: Can the Interrogator Tell?\n모방 게임: 심문자가 구분할 수 있는가?")
    ax1.set_ylim(40, 100)
    ax1.legend(loc="upper right")
    ax1.grid(True, alpha=0.3)

    # Right: Single experiment simulation
    ax2 = axes[1]
    n_trials = 100
    # Simulate 2024-level AI: interrogator ~55% accurate
    p_correct = 0.55
    results = np.random.binomial(1, p_correct, n_trials)

    cumulative_acc = np.cumsum(results) / np.arange(1, n_trials + 1)

    ax2.plot(range(1, n_trials + 1), cumulative_acc * 100, "darkorange", lw=2)
    ax2.axhline(50, color="gray", ls=":", lw=1, label="Chance (50%)")
    ax2.axhline(70, color="red", ls="--", lw=2, label="Turing threshold (70%)")
    ax2.fill_between(range(1, n_trials + 1),
                     45, 55, alpha=0.15, color="gray", label="Indistinguishable zone")

    ax2.set_xlabel("Number of trials / 시행 횟수")
    ax2.set_ylabel("Cumulative accuracy (%) / 누적 정확도 (%)")
    ax2.set_title("Simulated Turing Test (2024-level AI)\n시뮬레이션된 Turing Test (2024년 수준 AI)")
    ax2.set_ylim(40, 80)
    ax2.legend(loc="upper right", fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("Turing's criterion: if accuracy ≤ 70%, machine 'passes'")
    print("Chance level (50%): machine is indistinguishable from human")
    print(f"\nSimulated 2024 AI: {np.mean(results)*100:.0f}% detection rate over {n_trials} trials")

simulate_imitation_game()

## Part 5: Child Machine — Learning with Reward/Punishment / 어린이 기계: 보상/벌 학습

Turing's most prescient idea: instead of programming an adult mind, build a
**child machine** that learns through reward and punishment.

> "The machine has to be so constructed that events which shortly preceded
> a punishment signal are unlikely to be repeated, whereas a reward signal
> increased the probability of repetition." — Turing, Section 7

This is **reinforcement learning**, described 48 years before Sutton & Barto's textbook!
We implement a simple RL agent that learns to navigate a grid.

Turing의 가장 선견지명적인 아이디어: 성인의 지능을 프로그래밍하는 대신,
보상과 벌을 통해 학습하는 **어린이 기계**를 만든다.
이것은 Sutton & Barto의 교과서(1998)보다 48년 앞선 **강화학습** 기술입니다!

In [ ]:
class ChildMachine:
    """Turing's 'child machine' — learns through reward and punishment.

    A simple Q-learning agent on a grid world. The agent starts knowing
    nothing (blank notebook) and learns through experience.

    Turing's mapping:
        - Initial state (child brain) = Q-table initialized to zeros
        - Education = episodes of trial-and-error
        - Reward signal = positive reinforcement
        - Punishment signal = negative reinforcement
        - Random element = epsilon-greedy exploration
    """

    def __init__(self, grid_size, goal_pos, trap_positions=None,
                 learning_rate=0.1, discount=0.95, epsilon=0.3):
        """Initialize the child machine (knows nothing).

        Args:
            grid_size: Size of the square grid world.
            goal_pos: (row, col) of the goal (reward).
            trap_positions: List of (row, col) positions (punishment).
            learning_rate: How quickly the machine learns.
            discount: How much it values future rewards.
            epsilon: Probability of random exploration.
        """
        self.size = grid_size
        self.goal = goal_pos
        self.traps = set(trap_positions or [])
        self.lr = learning_rate
        self.gamma = discount
        self.epsilon = epsilon

        # "Rather little mechanism, and lots of blank sheets" — Turing
        # Q-table: the "notebook" that starts blank (all zeros)
        self.actions = [(0, 1), (0, -1), (1, 0), (-1, 0)]  # R, L, D, U
        self.action_names = ["→", "←", "↓", "↑"]
        self.q_table = np.zeros((grid_size, grid_size, len(self.actions)))

        self.episode_rewards = []

    def choose_action(self, state):
        """Choose action using epsilon-greedy (Turing's 'random element')."""
        if np.random.random() < self.epsilon:
            return np.random.randint(len(self.actions))  # explore randomly
        return np.argmax(self.q_table[state[0], state[1]])  # exploit knowledge

    def step(self, state, action_idx):
        """Take action, get reward/punishment."""
        dr, dc = self.actions[action_idx]
        new_r = np.clip(state[0] + dr, 0, self.size - 1)
        new_c = np.clip(state[1] + dc, 0, self.size - 1)
        new_state = (new_r, new_c)

        # Reward and punishment signals (Turing Section 7)
        if new_state == self.goal:
            reward = 10.0   # "reward signal"
        elif new_state in self.traps:
            reward = -5.0   # "punishment signal"
        else:
            reward = -0.1   # small cost to encourage efficiency

        return new_state, reward

    def learn_episode(self, start=(0, 0), max_steps=100):
        """One episode of learning (one 'lesson' in Turing's terms)."""
        state = start
        total_reward = 0

        for _ in range(max_steps):
            action = self.choose_action(state)
            next_state, reward = self.step(state, action)

            # Q-learning update: adjust based on reward/punishment
            # "Events preceding punishment become less likely to repeat"
            # "Events preceding reward become more likely to repeat"
            old_q = self.q_table[state[0], state[1], action]
            best_future = np.max(self.q_table[next_state[0], next_state[1]])
            new_q = old_q + self.lr * (reward + self.gamma * best_future - old_q)
            self.q_table[state[0], state[1], action] = new_q

            total_reward += reward
            state = next_state

            if state == self.goal:
                break

        self.episode_rewards.append(total_reward)
        return total_reward

    def train(self, n_episodes=500):
        """Educate the child machine over many episodes."""
        for ep in range(n_episodes):
            # Decay exploration over time (child becomes more confident)
            self.epsilon = max(0.01, 0.3 * (1 - ep / n_episodes))
            self.learn_episode()

    def get_policy(self):
        """Extract the learned policy (what the 'adult machine' knows)."""
        policy = np.full((self.size, self.size), " ")
        for r in range(self.size):
            for c in range(self.size):
                if (r, c) == self.goal:
                    policy[r, c] = "★"
                elif (r, c) in self.traps:
                    policy[r, c] = "✗"
                else:
                    best_action = np.argmax(self.q_table[r, c])
                    policy[r, c] = self.action_names[best_action]
        return policy


# Create and train the child machine
grid_size = 5
goal = (4, 4)
traps = [(1, 2), (2, 3), (3, 1)]

child = ChildMachine(grid_size, goal, traps)

print("BEFORE education (blank notebook) / 교육 전 (빈 노트북):")
print("Policy (random — knows nothing):")
policy_before = child.get_policy()
for row in policy_before:
    print("  " + "  ".join(row))

# Educate!
child.train(n_episodes=500)

print("\nAFTER education (500 episodes) / 교육 후 (500 에피소드):")
print("Policy (learned — navigates to goal, avoids traps):")
policy_after = child.get_policy()
for row in policy_after:
    print("  " + "  ".join(row))

print("\nLegend: ★=Goal(보상), ✗=Trap(벌), →↓←↑=Learned direction(학습된 방향)")

In [ ]:
def plot_learning_curve():
    """Visualize the child machine's learning progress."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Learning curve (reward over episodes)
    ax1 = axes[0]
    rewards = child.episode_rewards
    window = 20
    smoothed = np.convolve(rewards, np.ones(window) / window, mode="valid")

    ax1.plot(rewards, alpha=0.2, color="steelblue")
    ax1.plot(range(window - 1, len(rewards)), smoothed, "steelblue", lw=2,
             label=f"Moving avg ({window} episodes)")
    ax1.axhline(0, color="gray", ls=":", lw=0.5)
    ax1.set_xlabel("Episode (lesson) / 에피소드 (수업)")
    ax1.set_ylabel("Total reward / 총 보상")
    ax1.set_title("Child Machine Learning Curve\n어린이 기계 학습 곡선")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Annotate phases
    ax1.annotate("Random exploration\n(blank notebook)\n무작위 탐색 (빈 노트북)",
                 xy=(25, rewards[25]), xytext=(80, min(rewards) * 0.6),
                 arrowprops=dict(arrowstyle="->"), fontsize=9)
    ax1.annotate("Learned behavior\n(educated adult)\n학습된 행동 (교육된 성인)",
                 xy=(450, smoothed[-1]), xytext=(300, max(smoothed) * 0.7),
                 arrowprops=dict(arrowstyle="->"), fontsize=9)

    # Right: Q-value heatmap (what the machine "knows")
    ax2 = axes[1]
    max_q = np.max(child.q_table, axis=2)
    im = ax2.imshow(max_q, cmap="YlOrRd", origin="upper")
    plt.colorbar(im, ax=ax2, label="Max Q-value / 최대 Q값")

    # Overlay policy arrows and markers
    policy = child.get_policy()
    for r in range(child.size):
        for c in range(child.size):
            ax2.text(c, r, policy[r, c], ha="center", va="center",
                     fontsize=16, fontweight="bold",
                     color="white" if max_q[r, c] > max_q.max() * 0.5 else "black")

    ax2.set_title("Learned Knowledge (Q-values)\n학습된 지식 (Q값)")
    ax2.set_xlabel("Column")
    ax2.set_ylabel("Row")

    plt.tight_layout()
    plt.show()

    # Turing's mapping
    print("Turing's Child Machine → Modern RL Mapping:")
    print("=" * 50)
    print(f"  Child brain (blank)  → Q-table (all zeros)")
    print(f"  Education (500 eps)  → Training episodes")
    print(f"  Reward signal        → +10 at goal")
    print(f"  Punishment signal    → -5 at traps")
    print(f"  Random element       → ε-greedy exploration")
    print(f"  Adult mind (learned) → Optimal policy")

plot_learning_curve()

## Part 6: Connection to Modern AI / 현대 AI와의 연결

Turing's 1950 ideas map remarkably well to modern AI:

| Turing (1950) | Modern equivalent / 현대적 대응 |
|---|---|
| Child machine | Neural network (random init) |
| Blank notebook (Store) | Weight matrices (all random) |
| Education | Pre-training on data |
| Reward/punishment | RLHF (Reinforcement Learning from Human Feedback) |
| Random element | SGD, dropout, temperature sampling |
| Imperatives from teacher | System prompts, instruction tuning |
| Adult mind | Fine-tuned, deployed model |

The modern LLM training pipeline is essentially Turing's vision realized:

$$\text{Child machine} \xrightarrow{\text{pre-training (education)}} \text{Base model} \xrightarrow{\text{RLHF (reward/punishment)}} \text{Aligned model}$$

In [ ]:
def plot_turing_to_modern_timeline():
    """Visualize the path from Turing's ideas to modern AI."""
    fig, ax = plt.subplots(figsize=(16, 8))

    # Timeline data: (year, y_position, label, color)
    events = [
        (1950, 1, "Turing: Imitation Game\nChild machine, reward/punishment", "red"),
        (1956, 2, "Dartmouth Conference\n'AI' coined", "orange"),
        (1958, 1, "Rosenblatt: Perceptron\n(first learning machine)", "steelblue"),
        (1966, 2, "ELIZA\n(first chatbot)", "orange"),
        (1969, 1, "Minsky & Papert\nPerceptron limits → AI Winter", "gray"),
        (1986, 1, "Backpropagation\n(Rumelhart et al.)", "steelblue"),
        (1992, 2, "Watkins: Q-Learning\n(Turing's reward/punishment formalized)", "green"),
        (1997, 1, "Deep Blue\nChess champion", "steelblue"),
        (1998, 2, "Sutton & Barto\nRL textbook", "green"),
        (2012, 1, "AlexNet\nDeep learning revolution", "steelblue"),
        (2017, 1, "Transformer\n(Vaswani et al.)", "steelblue"),
        (2020, 2, "GPT-3\n(Brown et al.)", "purple"),
        (2022, 1, "InstructGPT / RLHF\nTuring's vision realized!", "red"),
        (2022.5, 2, "ChatGPT\nImitation Game at scale", "red"),
    ]

    for year, y_pos, label, color in events:
        ax.plot(year, y_pos, "o", color=color, ms=10, zorder=5)
        va = "bottom" if y_pos == 1 else "top"
        offset = 0.15 if y_pos == 1 else -0.15
        ax.annotate(label, (year, y_pos + offset), ha="center", va=va,
                    fontsize=8, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow",
                              ec=color, alpha=0.9))

    # Timeline axis
    ax.axhline(1.5, color="black", lw=2, zorder=1)
    ax.set_xlim(1945, 2028)
    ax.set_ylim(0, 3)
    ax.set_xlabel("Year / 연도", fontsize=12)
    ax.set_title("From Turing (1950) to ChatGPT (2022): Ideas Take 72 Years to Realize\n"
                 "Turing(1950)에서 ChatGPT(2022)까지: 아이디어가 실현되기까지 72년",
                 fontsize=13)
    ax.set_yticks([])

    # Color legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="red", ms=10, label="Turing's direct ideas"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="steelblue", ms=10, label="Neural network / DL milestones"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="green", ms=10, label="Reinforcement learning"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="purple", ms=10, label="Large language models"),
    ]
    ax.legend(handles=legend_elements, loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.show()

plot_turing_to_modern_timeline()

## Summary / 요약

| Part | What we built / 구현한 것 | Turing's concept / Turing의 개념 |
|------|--------------------------|--------------------------------|
| 1 | Discrete-state machine (turnstile) | Section 4: Digital computers are discrete-state machines / 디지털 컴퓨터는 이산 상태 기계 |
| 2 | Turing Machine from scratch | Section 4: Store + Executive + Control / 저장소 + 실행장치 + 제어 |
| 3 | Universal TM (one machine, many programs) | Section 5: Universality — intelligence is a software problem / 범용성 — 지능은 소프트웨어 문제 |
| 4 | Imitation Game statistics | Section 1–2: Turing's prediction and operational test / Turing의 예측과 조작적 테스트 |
| 5 | Child machine with Q-learning | Section 7: Learning through reward/punishment / 보상/벌을 통한 학습 |
| 6 | Turing → modern AI timeline | The 72-year arc from idea to realization / 아이디어에서 실현까지 72년의 궤적 |

**Next paper / 다음 논문**: #3 Rosenblatt (1958) — The Perceptron.
Turing's "child machine" vision gets its first concrete implementation:
a machine that actually *learns from data*.

다음 논문: #3 Rosenblatt (1958) — Perceptron.
Turing의 "어린이 기계" 비전이 최초로 구체적으로 구현됩니다:
실제로 *데이터로부터 학습하는* 기계.